# Customer Service Chatbot: Learning Walkthrough
Personal learning portfolio, independently assembled with ChatGPT assistance for learning, coding and debugging. All example messages are simulated; no employer systems or customer records are needed.

Extract the complete project and open this notebook with its folder as the working directory. Run cells from top to bottom. In Colab, upload the project files into the current working directory first. This notebook imports the Python modules rather than overwriting them. Only QA result files and the dashboard are regenerated.

Final review executed all six code cells sequentially in a fresh Python process. A full Jupyter/Colab kernel and browser session was not tested. Saved outputs and execution counts are deliberately empty.


In [ ]:
from pathlib import Path
import sys

BASE = Path.cwd()
if not (BASE / "intent_classifier.py").is_file():
    raise FileNotFoundError("Open the notebook from the extracted project folder.")
sys.path.insert(0, str(BASE))
from intent_classifier import classify, evaluate
from test_chatbot import run_tests
import chatbot


## 1. Intent classification
Rules are checked in order. The first matching intent wins. The 60 original examples are known development/regression cases, not a held-out benchmark.

In [ ]:
intent_results = evaluate(BASE / "test_cases.csv", BASE / "qa_results.csv")


## 2. Additional original examples
These labels were recovered unchanged from the original notebook. They were seen during development and are not evidence of unseen real-world accuracy. Brand wording alone was made generic.

In [ ]:
extra_results = {}
for name in ["development", "validation", "additional"]:
    extra_results[name] = evaluate(BASE / f"{name}_cases.csv", BASE / f"{name}_results.csv")


## 3. Multi-turn clarification
A charge enquiry sets a waiting state. A choice resolves it; another supported topic switches the conversation. Cancel and restart clear the state.

In [ ]:
chatbot.conversation_state = None
for message in ["I have a charge issue", "2", "My internet is not working", "cancel"]:
    print("Customer:", message)
    print("Bot:", chatbot.chat(message))
conversation_results = run_tests(BASE / "conversation_qa_results.csv")


## 4. Edge-case observations
Direct device charging is deliberately outside the classifier categories and returns Fallback. The charging response is only available after clarification. Negation is not parsed, and a mixed message returns only one intent. These are limitations, not successful full resolutions.

In [ ]:
edge_cases = [
    "It's not about my bill, my phone won't charge",
    "I don't have a network issue, I forgot my password",
    "I can't log in and my internet is not working",
    "My bill is fine, but I can't place an online order",
]
for message in edge_cases:
    chatbot.conversation_state = None
    print(message, "->", classify(message), "|", chatbot.chat(message))
chatbot.conversation_state = None


## 5. Reproducible QA dashboard
The dashboard uses the regenerated result CSVs for all 106 predefined examples. These are previously seen regression cases, not unseen accuracy. The separate engineering checks in `test_review.py` are not included in this denominator.


In [ ]:
from run_qa import write_dashboard
write_dashboard(BASE)
print((BASE / "qa_dashboard.svg").name)
